# Developing final bakery classifier

This notebook aims to create the final bakery business name classifier using the labels combined from the OSM and Companies House data, as well comparing varying modelling strategies.

In [89]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline


OSM_DATA_DATE = "2026-07-27"
COMPANIES_HOUSE_DATA_DATE = "2026-07-28"
FHRS_DATA_DATE = "2026-07-23"

INTERIM_FOLDER = Path("../data/business/interim")
INTERIM_FOLDER.mkdir(parents=True, exist_ok=True)

MODEL_FOLDER = Path("../models")
MODEL_FOLDER.mkdir(parents=True, exist_ok=True)

OSM_TRAINING_PATH = (INTERIM_FOLDER/ f"england_osm_training_names_{OSM_DATA_DATE}.csv")

COMPANIES_HOUSE_TRAINING_PATH = (INTERIM_FOLDER/ f"companies_house_training_names_{COMPANIES_HOUSE_DATA_DATE}.csv")

FHRS_PATH = (INTERIM_FOLDER/ f"london_fhrs_prepared_{FHRS_DATA_DATE}.csv")


osm_training = pd.read_csv(OSM_TRAINING_PATH, usecols=["DisplayName", "CleanBusinessName", "BakeryLabel"])

companies_house_training = pd.read_csv(COMPANIES_HOUSE_TRAINING_PATH, usecols=["DisplayName", "CleanBusinessName", "BakeryLabel"])

fhrs = pd.read_csv(FHRS_PATH)

print(f"OSM names: {len(osm_training)} \nCompanies House names: {len(companies_house_training)} \nFHRS establishments: {len(fhrs)}")

OSM names: 82894 
Companies House names: 49591 
FHRS establishments: 81080


## Splitting datasets for training

The aim here will be to produce multiple models and compare them to a benchmark. This will be done by splitting both osm and companies house datasets into training and testing, then creating a osm only training model, a companies house only training model, a shared training model with balanced and unbalanced weights, and a character vs character and word model.


In [60]:
osm_model_data = osm_training[["CleanBusinessName", "BakeryLabel"]].copy()
companies_house_model_data = companies_house_training[["CleanBusinessName", "BakeryLabel"]].copy()

# Remove names receiving different labels across the two sources.
combined_model_data = pd.concat([osm_model_data, companies_house_model_data], ignore_index=True)

labels_per_name = (combined_model_data.groupby("CleanBusinessName")["BakeryLabel"].nunique())
conflicting_names = (labels_per_name[labels_per_name > 1].index)

osm_model_data = osm_model_data[~osm_model_data["CleanBusinessName"].isin(conflicting_names)].copy()
companies_house_model_data = companies_house_model_data[~companies_house_model_data["CleanBusinessName"].isin(conflicting_names)].copy()

# Split each source separately.
osm_train, osm_test = train_test_split(osm_model_data,
    test_size=0.20,
    random_state=2026,
    stratify=osm_model_data["BakeryLabel"])

companies_house_train, companies_house_test = (
    train_test_split(companies_house_model_data,
        test_size=0.20,
        random_state=2026,
        stratify=companies_house_model_data["BakeryLabel"]))

# Prevent names in either testing set appearing in training.
testing_business_names = (set(osm_test["CleanBusinessName"]) | set(companies_house_test["CleanBusinessName"]))

osm_train = osm_train[~osm_train["CleanBusinessName"].isin(testing_business_names)].reset_index(drop=True)
companies_house_train = companies_house_train[~companies_house_train["CleanBusinessName"].isin(testing_business_names)].reset_index(drop=True)

# Create combined datasets with one row per cleaned name.
combined_training_dataset = (pd.concat([osm_train, companies_house_train], ignore_index=True)
    .drop_duplicates(subset="CleanBusinessName")
    .reset_index(drop=True))

combined_testing_dataset = (pd.concat([osm_test, companies_house_test], ignore_index=True)
    .drop_duplicates(subset="CleanBusinessName")
    .reset_index(drop=True))


print(f"Conflicting names removed:{len(conflicting_names)}\n"
    f"OSM train / test count: {len(osm_train)} / {len(osm_test)}\n"
    f"Companies House train / test couunt: {len(companies_house_train)} / {len(companies_house_test)}\n"
    f"Combined train / test count: {len(combined_training_dataset)} / {len(combined_testing_dataset)}")

Conflicting names removed:0
OSM train / test count: 66312 / 16579
Companies House train / test couunt: 39671 / 9919
Combined train / test count: 105972 / 26498


# Defining model features

The model configurations will be varied between the OSM-only, Companies house-only and combined models with changes in the character/word features in the TF-IDF and the class weights.

In [30]:
def build_bakery_model(feature_type, class_weight):
    character_features = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)

    if feature_type == "character":
        features = character_features

    else:
        features = FeatureUnion([
            ("character_tfidf", character_features),
            ("word_tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2))
            ])

    return Pipeline([
        ("features", features),
        ("classifier", LogisticRegression(class_weight=class_weight, max_iter=1000, random_state=42))
        ])

# Defining model options

There are 12 different models that are worthwhile to compare with these differences in features.

These are:
1. OSM + character + balanced
2. OSM + character + unbalanced
3. OSM + character and word + balanced
4. OSM + character and word + unbalanced
5. Companies house + character + balanced
6. Companies house + character + unbalanced
7. Companies house + character and word + balanced
8. Companies house + character and word + unbalanced
9. Combined + character + balanced
10. Combined + character + unbalanced
11. Combined + character and word + balanced
12. Combined + character and word + unbalanced

In [31]:
model_options = {
    "OSM | character | balanced": {
        "source": "OSM",
        "training_data": osm_train,
        "feature_type": "character",
        "class_weight": "balanced"},

    "OSM | character | unweighted": {
        "source": "OSM",
        "training_data": osm_train,
        "feature_type": "character",
        "class_weight": None},

    "OSM | character and word | balanced": {
        "source": "OSM",
        "training_data": osm_train,
        "feature_type": "character_and_word",
        "class_weight": "balanced"},

    "OSM | character and word | unweighted": {
        "source": "OSM",
        "training_data": osm_train,
        "feature_type": "character_and_word",
        "class_weight": None},

    "Companies House | character | balanced": {
        "source": "Companies House",
        "training_data": companies_house_train,
        "feature_type": "character",
        "class_weight": "balanced"},

    "Companies House | character | unweighted": {
        "source": "Companies House",
        "training_data": companies_house_train,
        "feature_type": "character",
        "class_weight": None},

    "Companies House | character and word | balanced": {
        "source": "Companies House",
        "training_data": companies_house_train,
        "feature_type": "character_and_word",
        "class_weight": "balanced"},

    "Companies House | character and word | unweighted": {
        "source": "Companies House",
        "training_data": companies_house_train,
        "feature_type": "character_and_word",
        "class_weight": None},

    "Combined | character | balanced": {
        "source": "Combined",
        "training_data": combined_training_dataset,
        "feature_type": "character",
        "class_weight": "balanced"},

    "Combined | character | unweighted": {
        "source": "Combined",
        "training_data": combined_training_dataset,
        "feature_type": "character",
        "class_weight": None},

    "Combined | character and word | balanced": {
        "source": "Combined",
        "training_data": combined_training_dataset,
        "feature_type": "character_and_word",
        "class_weight": "balanced"},

    "Combined | character and word | unweighted": {
        "source": "Combined",
        "training_data": combined_training_dataset,
        "feature_type": "character_and_word",
        "class_weight": None},
}

In [32]:
def evaluate_scores(actual_labels, scores, threshold):
    actual_labels = np.asarray(actual_labels)

    predictions = (scores >= threshold).astype("int64")

    true_positives = int(((predictions == 1) & (actual_labels == 1)).sum())
    false_positives = int(((predictions == 1) & (actual_labels == 0)).sum())
    false_negatives = int(((predictions == 0) & (actual_labels == 1)).sum())

    predicted_bakeries = (true_positives + false_positives)

    precision = (true_positives / predicted_bakeries )

    recall = (true_positives/ (true_positives+ false_negatives))

    return {
        "Precision": precision,
        "Recall": recall,
        "TruePositives": true_positives,
        "FalsePositives": false_positives,
        "FalseNegatives": false_negatives}

In [61]:
shared_development_names = (
    set(osm_train["CleanBusinessName"])
    & set(
        companies_house_train[
            "CleanBusinessName"
        ]
    )
)

osm_cv_data = osm_train[
    ~osm_train["CleanBusinessName"].isin(
        shared_development_names
    )
].copy()

companies_house_cv_data = companies_house_train[
    ~companies_house_train[
        "CleanBusinessName"
    ].isin(shared_development_names)
].copy()

osm_cv_data["ValidationSource"] = "OSM"

companies_house_cv_data[
    "ValidationSource"
] = "Companies House"

cv_dataset = (
    pd.concat(
        [
            osm_cv_data,
            companies_house_cv_data,
        ],
        ignore_index=True,
    )
    .reset_index(drop=True)
)

cv_dataset["CVStratum"] = (
    cv_dataset["ValidationSource"]
    + "_"
    + cv_dataset["BakeryLabel"].astype(str)
)

print(
    f"Shared names removed from cross-validation: "
    f"{len(shared_development_names):,}\n"
    f"Cross-validation names: "
    f"{len(cv_dataset):,}"
)

Shared names removed from cross-validation: 11
Cross-validation names: 105,961


In [62]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

oof_scores_by_model = {}


for model_name, settings in model_options.items():
    oof_scores = np.zeros(
        len(cv_dataset)
    )

    for train_indices, validation_indices in (
        cross_validation.split(
            cv_dataset,
            cv_dataset["CVStratum"],
        )
    ):
        fold_training_data = cv_dataset.iloc[
            train_indices
        ]

        fold_validation_data = cv_dataset.iloc[
            validation_indices
        ]

        if settings["source"] == "OSM":
            model_training_data = (
                fold_training_data[
                    fold_training_data[
                        "ValidationSource"
                    ].eq("OSM")
                ]
            )

        elif settings["source"] == (
            "Companies House"
        ):
            model_training_data = (
                fold_training_data[
                    fold_training_data[
                        "ValidationSource"
                    ].eq("Companies House")
                ]
            )

        else:
            model_training_data = (
                fold_training_data
            )

        model = build_bakery_model(
            feature_type=settings[
                "feature_type"
            ],
            class_weight=settings[
                "class_weight"
            ],
        )

        model.fit(
            model_training_data[
                "CleanBusinessName"
            ],
            model_training_data[
                "BakeryLabel"
            ],
        )

        oof_scores[
            validation_indices
        ] = model.predict_proba(
            fold_validation_data[
                "CleanBusinessName"
            ]
        )[:, 1]

    oof_scores_by_model[
        model_name
    ] = oof_scores


print(
    f"Cross-validation completed for "
    f"{len(oof_scores_by_model)} models."
)

Cross-validation completed for 12 models.


In [65]:
cv_result_rows = []

validation_sources = ["OSM", "Companies House", "Combined"]


for model_name, scores in (oof_scores_by_model.items()):
    for validation_source in (validation_sources):
        if validation_source == "Combined":
            source_mask = np.ones(len(cv_dataset),dtype=bool)

        else:
            source_mask = (cv_dataset["ValidationSource"].eq(validation_source).to_numpy())

        results = evaluate_scores(
            actual_labels=cv_dataset.loc[source_mask, "BakeryLabel"],
            scores=scores[source_mask],
            threshold=0.50)

        precision = results["Precision"]
        recall = results["Recall"]

        f1 = (2 * precision * recall / (precision + recall))

        cv_result_rows.append({
            "Model": model_name,
            "ValidationSource": (validation_source),
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
        })


cv_results = pd.DataFrame(
    cv_result_rows
)

In [66]:
comparison_rows = []

for model_name in model_options:
    row = {"Model": model_name}

    for validation_source in (validation_sources):
        result = cv_results[cv_results["Model"].eq(model_name) & cv_results["ValidationSource"].eq(validation_source)].iloc[0]

        row[f"{validation_source} Precision"] = result["Precision"]

        row[f"{validation_source} Recall"] = result["Recall"]

        row[f"{validation_source} F1"] = result["F1"]

    comparison_rows.append(row)


cv_model_comparison = pd.DataFrame(comparison_rows)

cv_model_comparison.round(3)

,Model,OSM Precision,OSM Recall,OSM F1,Companies House Precision,Companies House Recall,Companies House F1,Combined Precision,Combined Recall,Combined F1
0,OSM | character | balanced,0.447,0.763,0.564,0.525,0.413,0.462,0.493,0.498,0.496
1,OSM | character | unweighted,0.835,0.532,0.650,0.874,0.132,0.229,0.852,0.229,0.360
2,OSM | character and word | balanced,0.495,0.749,0.596,0.415,0.425,0.420,0.441,0.504,0.470
3,OSM | character and word | unweighted,0.838,0.551,0.665,0.883,0.156,0.265,0.859,0.251,0.389
4,Companies House | character | balanced,0.137,0.802,0.234,0.481,0.621,0.542,0.278,0.665,0.392
5,Companies House | character | unweighted,0.572,0.669,0.617,0.841,0.408,0.550,0.724,0.471,0.571
6,Companies House | character and word | balanced,0.162,0.780,0.268,0.498,0.617,0.551,0.312,0.656,0.423
7,Companies House | character and word | unweighted,0.535,0.676,0.597,0.836,0.422,0.561,0.702,0.483,0.573
8,Combined | character | balanced,0.511,0.748,0.607,0.297,0.745,0.425,0.331,0.746,0.458
9,Combined | character | unweighted,0.794,0.551,0.651,0.824,0.382,0.522,0.815,0.423,0.557


In [67]:
def find_recall_threshold(actual_labels, scores, target_recall=0.90):
    actual_labels = np.asarray(actual_labels)
    scores = np.asarray(scores)

    positive_scores = scores[actual_labels == 1]

    positive_scores = np.sort(positive_scores)[::-1]

    required_positives = int(np.ceil(target_recall * len(positive_scores)))

    return float(positive_scores[required_positives - 1])

In [72]:
def compare_oof_models_at_target_recall(target_recall):

    comparison_rows = []

    actual_labels = (cv_dataset["BakeryLabel"].to_numpy())
    osm_mask = (cv_dataset["ValidationSource"].eq("OSM").to_numpy())
    companies_house_mask = (cv_dataset["ValidationSource"].eq("Companies House").to_numpy())

    for model_name, scores in (oof_scores_by_model.items()):
        osm_threshold = find_recall_threshold(
            actual_labels=actual_labels[osm_mask],
            scores=scores[osm_mask],
            target_recall=target_recall)

        companies_house_threshold = (
            find_recall_threshold(
                actual_labels=actual_labels[companies_house_mask],
                scores=scores[companies_house_mask],
                target_recall=target_recall))

        # Use the lower threshold so that the recall target is achieved for both sources.
        model_threshold = min(osm_threshold, companies_house_threshold)

        osm_results = evaluate_scores(
            actual_labels=actual_labels[osm_mask],
            scores=scores[osm_mask],
            threshold=model_threshold)

        companies_house_results = (
            evaluate_scores(
                actual_labels=actual_labels[companies_house_mask],
                scores=scores[companies_house_mask],
                threshold=model_threshold))

        combined_results = evaluate_scores(
            actual_labels=actual_labels,
            scores=scores,
            threshold=model_threshold)

        comparison_rows.append({
            "Target Recall": target_recall,
            "Model": model_name,
            "Threshold": model_threshold,
            "OSM Precision": osm_results["Precision"],
            "OSM Recall": osm_results["Recall"],
            "Companies House Precision": (companies_house_results["Precision"]),
            "Companies House Recall": (companies_house_results["Recall"]),
            "Minimum Recall": min(osm_results["Recall"], companies_house_results["Recall"],),
            "Combined Precision": (combined_results["Precision"]),
            "Combined Recall": (combined_results["Recall"]),
            "Combined False Positives": (combined_results["FalsePositives"]),
            "Combined False Negatives": (combined_results["FalseNegatives"])
        })

    return (
        pd.DataFrame(comparison_rows)
        .sort_values(["Combined Precision", "Combined False Positives"], ascending=[False, True],
        ).reset_index(drop=True)
        )

In [74]:
target_recalls = [0.90, 0.95, 0.99]

high_recall_cv_results = pd.concat(
    [compare_oof_models_at_target_recall(target_recall) for target_recall in target_recalls], 
    ignore_index=True)

In [ ]:
top_models_by_recall = (high_recall_cv_results
    .groupby("Target Recall", group_keys=False)
    .head(5))

combined_negatives = int((cv_dataset["BakeryLabel"] == 0).sum())

high_recall_cv_results["False Positives per 1,000 Negatives"] = (
    high_recall_cv_results["Combined False Positives"] / combined_negatives * 1000
    )

top_models_by_recall[
    [
        "Target Recall",
        "Model",
        "Threshold",
        "OSM Precision",
        "OSM Recall",
        "Companies House Precision",
        "Companies House Recall",
        "Combined Precision",
        "Combined Recall",
        "Combined False Positives",
        "Combined False Negatives",
    ]].round(3)

,Target Recall,Model,Threshold,OSM Precision,OSM Recall,Companies House Precision,Companies House Recall,Combined Precision,Combined Recall,Combined False Positives,Combined False Negatives
0,0.90,Combined | character and word | balanced,0.089,0.115,0.900,0.161,0.978,0.147,0.959,37774,277
1,0.90,Combined | character and word | unweighted,0.021,0.117,0.900,0.160,0.981,0.147,0.961,37890,264
2,0.90,Combined | character | unweighted,0.022,0.107,0.900,0.154,0.984,0.140,0.964,40297,248
3,0.90,Combined | character | balanced,0.114,0.103,0.900,0.154,0.984,0.139,0.964,40766,247
4,0.90,Companies House | character and word | balanced,0.191,0.049,0.938,0.209,0.900,0.115,0.909,47850,619
12,0.95,Combined | character and word | unweighted,0.015,0.072,0.950,0.150,0.989,0.119,0.979,49267,141
13,0.95,Combined | character and word | balanced,0.050,0.068,0.950,0.148,0.991,0.116,0.981,50907,130
14,0.95,Combined | character | unweighted,0.015,0.061,0.950,0.146,0.992,0.110,0.982,54205,124
15,0.95,Combined | character | balanced,0.062,0.056,0.950,0.143,0.995,0.105,0.984,57008,110
16,0.95,OSM | character and word | unweighted,0.007,0.050,0.984,0.147,0.950,0.100,0.958,58998,285


In [ ]:
full_labelled_dataset = (pd.concat(
    [osm_model_data, companies_house_model_data], ignore_index=True)
    .drop_duplicates(subset="CleanBusinessName")
    .reset_index(drop=True))

final_bakery_model = build_bakery_model(feature_type="character_and_word", class_weight=None,)

final_bakery_model.fit(full_labelled_dataset["CleanBusinessName"], full_labelled_dataset["BakeryLabel"])

print(f"Final model trained using {len(full_labelled_dataset)} unique labelled names.")

Final model trained using 132470 unique labelled names.


In [100]:
fhrs_unique_names = (fhrs[fhrs["BusinessNameClean"].notna() & fhrs["BusinessNameClean"].ne("")
                          ].groupby("BusinessNameClean", as_index=False)
                          ).agg(DisplayName=("BusinessName", "first"), StoreCount=("FHRSID", "nunique"))

fhrs_unique_names["BakeryScore"] = (final_bakery_model.predict_proba(fhrs_unique_names["BusinessNameClean"])[:, 1])

fhrs_ranked_names = (fhrs_unique_names
                     .sort_values(["BakeryScore", "StoreCount"], ascending=[False, False])
                     .reset_index(drop=True))

fhrs_ranked_names["BakeryRank"] = np.arange(1, len(fhrs_ranked_names) + 1)

fhrs_ranked_names[["BakeryRank", "DisplayName", "StoreCount", "BakeryScore"]].head(50)

,BakeryRank,DisplayName,StoreCount,BakeryScore
0,1,Bakery And Cake,1,0.999579
1,2,G.O.D Cakes / Bakery,1,0.999578
2,3,Bakers Cakes,1,0.999565
3,4,Sweetsop Bakery,1,0.999538
4,5,Eggfree Cake Box (DJTC Cakes Ltd),1,0.999166
5,6,B.Bakery,1,0.999023
6,7,Q's Bakery,1,0.998964
7,8,Q's Bakery Ltd,1,0.998757
8,9,B.J Bakery,1,0.998572
9,10,SweetSmile Bakery,1,0.998348


In [105]:
fhrs_ranked_establishments = (
    fhrs.merge(fhrs_ranked_names[["BusinessNameClean", "BakeryScore", "BakeryRank", "StoreCount"]],
        on="BusinessNameClean",
        how="left")
    .sort_values(["BakeryRank", "FHRSID"], na_position="last")
    .reset_index(drop=True))

print(f"Unique ranked names: {len(fhrs_ranked_names)}\n")

print(f"Ranked FHRS establishments: {len(fhrs_ranked_establishments)}")

RANKED_NAMES_PATH = (INTERIM_FOLDER / f"london_fhrs_ranked_names_{FHRS_DATA_DATE}.csv")

RANKED_ESTABLISHMENTS_PATH = (INTERIM_FOLDER / f"london_fhrs_ranked_establishments_{FHRS_DATA_DATE}.csv")

FINAL_MODEL_PATH = (MODEL_FOLDER / "final_bakery_screening_model.joblib")

fhrs_ranked_names.to_csv(RANKED_NAMES_PATH, index=False,)

fhrs_ranked_establishments.to_csv(RANKED_ESTABLISHMENTS_PATH, index=False)

joblib.dump(final_bakery_model, FINAL_MODEL_PATH)

print(f"\nRanked names saved to: {RANKED_NAMES_PATH}")

print(f"Ranked establishments saved to: {RANKED_ESTABLISHMENTS_PATH}")

print(f"Final model saved to: {FINAL_MODEL_PATH}")

Unique ranked names: 65799

Ranked FHRS establishments: 81080

Ranked names saved to: ..\data\business\interim\london_fhrs_ranked_names_2026-07-23.csv
Ranked establishments saved to: ..\data\business\interim\london_fhrs_ranked_establishments_2026-07-23.csv
Final model saved to: ..\models\final_bakery_screening_model.joblib
